In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
import joblib

grid = pd.read_parquet('../data/processed/daily_sales.parquet')
print(f'Grid shape: {grid.shape}')

Grid shape: (1754, 15972)


In [3]:
LAG_DAYS     = [1, 2, 3, 7, 14, 21, 28, 35, 42, 56]
ROLL_WINDOWS = [7, 14, 28, 56, 90]
DOW_LAG_WKS  = [1, 2, 4, 8, 12]

def build_features(d, series):
    hist = series[series.index < d]
    row  = {
        'dow'           : d.dayofweek,
        'month'         : d.month,
        'dom'           : d.day,
        'quarter'       : d.quarter,
        'week'          : int(d.isocalendar().week),
        'is_weekend'    : int(d.dayofweek >= 5),
        'is_tet'        : int((d.month==1 and d.day>=15) or
                              (d.month==2 and d.day<=15)),
        'is_month_end'  : int(d.day >= 25),
        'is_month_start': int(d.day <= 5),
    }
    for lag in LAG_DAYS:
        row[f'lag_{lag}'] = float(
            series.get(d - pd.Timedelta(days=lag), 0.0))

    for w in ROLL_WINDOWS:
        win = hist.iloc[-w:] if len(hist) >= w else hist
        row[f'rmean_{w}'] = float(win.mean())
        row[f'rstd_{w}']  = float(win.std()) if len(win) > 1 else 0.
        row[f'rmax_{w}']  = float(win.max())
        row[f'rpos_{w}']  = float((win > 0).mean())

    same_dow = hist[hist.index.dayofweek == d.dayofweek]
    for wk in DOW_LAG_WKS:
        row[f'dlag_{wk}w'] = float(same_dow.iloc[-wk]) \
                              if len(same_dow) >= wk else 0.

    t28 = hist.iloc[-28:]
    row['trend_28'] = float(
        np.polyfit(np.arange(len(t28)), t28.values.astype(float), 1)[0]
    ) if len(t28) > 2 else 0.

    row['zero_frac_28'] = float((t28 == 0).mean()) if len(t28) > 0 else 1.

    ly     = d - pd.DateOffset(years=1)
    ly_win = hist[(hist.index >= ly - pd.Timedelta(days=14)) &
                  (hist.index <= ly + pd.Timedelta(days=14))]
    row['ly_mean'] = float(ly_win.mean()) if len(ly_win) > 0 else 0.

    return row

print(f'build_features defined — {len(build_features(pd.Timestamp("2025-08-01"), grid[grid.columns[0]]))} features')

build_features defined — 47 features


In [4]:
qty_90d  = grid[grid.index >= '2025-06-07'].sum()
qty_365d = grid[grid.index >= '2024-09-06'].sum()

TIER_A = qty_90d[(qty_90d >= 150) & (qty_365d >= 400)].index.tolist()
TIER_B = qty_90d[(qty_90d >= 30)  & (qty_365d >= 80) &
                 (~qty_90d.index.isin(TIER_A))].index.tolist()

print(f'Tier A : {len(TIER_A)} SKUs')
print(f'Tier B : {len(TIER_B)} SKUs')

Tier A : 128 SKUs
Tier B : 376 SKUs


In [5]:
X_rows, y_rows = [], []

for i, sku in enumerate(TIER_A):
    series = grid[sku]
    for d in pd.date_range('2024-03-01', '2025-09-04', freq='2D'):
        X_rows.append(build_features(d, series))
        y_rows.append(float(series.get(d, 0.0)))

    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(TIER_A)} SKUs processed — {len(X_rows):,} rows so far')

X = pd.DataFrame(X_rows).fillna(0)
y = np.log1p(y_rows)

print(f'\nTraining samples : {len(X):,}')
print(f'Features         : {X.shape[1]}')

  20/128 SKUs processed — 5,540 rows so far
  40/128 SKUs processed — 11,080 rows so far
  60/128 SKUs processed — 16,620 rows so far
  80/128 SKUs processed — 22,160 rows so far
  100/128 SKUs processed — 27,700 rows so far
  120/128 SKUs processed — 33,240 rows so far

Training samples : 35,456
Features         : 47


In [6]:
model = HistGradientBoostingRegressor(
    max_iter          = 500,
    learning_rate     = 0.04,
    max_leaf_nodes    = 47,
    min_samples_leaf  = 15,
    l2_regularization = 0.5,
    random_state      = 42
)
model.fit(X, y)

pred_is = np.maximum(np.expm1(model.predict(X)), 0)
print(f'In-sample MAE : {np.mean(np.abs(pred_is - np.array(y_rows))):.3f}')

joblib.dump(model, '../models/hgbr_model.pkl')
print('Saved → ../models/hgbr_model.pkl')

In-sample MAE : 5.210
Saved → ../models/hgbr_model.pkl


In [7]:
VAL_DATES  = pd.date_range('2025-08-09', '2025-09-05', freq='D')
train_grid = grid[grid.index < '2025-08-09']
weights    = train_grid.iloc[-28:].sum()
actual     = grid.loc[VAL_DATES]

FEAT_COLS = X.columns.tolist()

val_preds = {}
for sku in TIER_A:
    series = grid[sku].copy()
    preds  = []
    for d in VAL_DATES:
        row  = build_features(d, series[series.index < d])
        Xrow = pd.DataFrame([row], columns=FEAT_COLS).fillna(0)
        pred = max(0.0, float(np.expm1(model.predict(Xrow)[0])))
        preds.append(pred)
        series[d] = pred
    val_preds[sku] = np.array(preds)

pred_ml = pd.DataFrame(0.0, index=VAL_DATES, columns=grid.columns)
for sku in TIER_A:
    pred_ml[sku] = val_preds[sku]

def wrmsse(pred, actual, train, weights):
    mse        = ((pred.values - actual.values) ** 2).mean(axis=0)
    train_mean = train.iloc[-28:].mean(axis=0).values
    scale      = train_mean ** 2 + 1e-8
    rmsse      = np.sqrt(mse / scale)
    w          = weights.values / weights.sum()
    return float((w * rmsse).sum())

score = wrmsse(pred_ml, actual, train_grid, weights)
print(f'ML model WRMSSE (Tier A only) : {round(score, 4)}')
print(f'Baseline to beat              : 2.2522')
print(f'Improvement                   : {round(2.2522 - score, 4)}')

ML model WRMSSE (Tier A only) : 2.0656
Baseline to beat              : 2.2522
Improvement                   : 0.1866
